In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_dna_member.generate_population as gp
import lib_dna_member.job_manager as job_manager
import lib_dna_member.coupon_digital_features as features
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:
def generate_coupon_and_digital(job):
    """
    Generate the features associated with coupon and digital

    Parameters:
        job (managers.JobManager): object which manages the Spark App
    Returns:
        (pyspark.sql.DataFrame): dna with new features
    """
    dna = job.tables["cubes_coupon_and_digital_1"].select(
        "MBRSHP_SID",
        "FISCAL_WEEK_END",
        "FW_COUPON_REDEMPTIONS",
        "FW_COUPON_REDEMPTIONS_W_CLPLSS",
        "FW_COUPON_SAVINGS",
        "FW_COUPON_SAVINGS_W_CLPLSS",
    )

    orig_cols = dna.columns

    # Compute the aggregate feature per customer {weeks} fiscal weeks back
    weeks = ["FOUR", "EIGHT", "TWELVE", "TWENTY-SIX", "FIFTY-TWO"]
    for num_weeks in weeks:
        dna = features.feature_aggregate_per_member_weeks(
            job, dna, num_weeks, "FW_COUPON_REDEMPTIONS"
        )

        dna = features.feature_aggregate_per_member_weeks(
            job, dna, num_weeks, "FW_COUPON_REDEMPTIONS_W_CLPLSS"
        )

        dna = features.feature_aggregate_per_member_weeks(
            job, dna, num_weeks, "FW_COUPON_SAVINGS"
        )

        dna = features.feature_aggregate_per_member_weeks(
            job, dna, num_weeks, "FW_COUPON_SAVINGS_W_CLPLSS"
        )

        # dna = utils.cache_df(dna)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get(
    "recency_lookback_duration", {}
)
member_dna_input_data_validator(
    silver_skeleton, silver_master_member_extended, fs_cubes_coupon_and_digital_1,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)


In [0]:
job.read_table("skeleton") 
job.read_table("member_extended")
job.read_table("cubes_coupon_and_digital_1")

In [0]:
features = generate_coupon_and_digital(job)

### Save results

In [0]:
spark.sql(f"DELETE FROM {fs_cubes_coupon_and_digital_2}")

fe = FeatureEngineeringClient()

fe.write_table(
    name=fs_cubes_coupon_and_digital_2,
    df=features,
    mode="merge"
)